# Clean and Integrate Golf Putting Data

This notebook cleans and integrates two golf datasets for the IS477 course project:

- **Dataset 1:** 2025 PGA Tour SG: Putting
- **Dataset 2:** 2026 Masters leaderboard + putter data

It produces:

- `data/cleaned/pga_sgputt_2025_clean.csv`
- `data/cleaned/masters_2026_clean.csv`
- `data/integrated/integrated_putter_analysis.csv`

It also creates a log file summarizing what happened during the run.

In [2]:
import os
import hashlib
from datetime import datetime

import pandas as pd

In [5]:
# Base folders
RAW_DIR = os.path.join(".")
CLEAN_DIR = os.path.join(".", "data", "cleaned")
INT_DIR = os.path.join(".", "data", "integrated")
LOG_DIR = os.path.join(".", "logs")

# Make sure output folders exist
for folder in [CLEAN_DIR, INT_DIR, LOG_DIR]:
    os.makedirs(folder, exist_ok=True)

# Collect log messages as we go
from datetime import datetime, timezone
log_lines = [f"=== Notebook run: {datetime.now(timezone.utc).isoformat()} UTC ===\n"]


def log(message):
    print(message)
    log_lines.append(str(message))


def sha256(file_path):
    """Return SHA-256 hash of a file."""
    hasher = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

## Helper function for putter era

This turns a putter release year into a simple category that may be useful later in analysis.

In [6]:
def classify_era(year):
    if pd.isna(year):
        return "unknown"
    
    year = int(year)
    
    if year <= 2015:
        return "classic (pre-2016)"
    elif year <= 2020:
        return "mid-era (2016-2020)"
    else:
        return "modern (2021+)"

## Dataset 1: Clean 2025 PGA Tour SG: Putting data

In [7]:
log("\n── Dataset 1: 2025 PGA Tour SG:Putting ──")

pga_path = os.path.join(RAW_DIR, "pga_tour_sgputt_2025_raw.csv")
pga = pd.read_csv(pga_path)

log(f"Loaded {len(pga)} rows, {len(pga.columns)} columns")
pga.head()


── Dataset 1: 2025 PGA Tour SG:Putting ──
Loaded 40 rows, 10 columns


,rank,player_name,sg_putting_avg,putter_type,putter_brand,putter_model,putter_release_year,group,season,source
0,1,Sam Burns,0.983,mallet,Odyssey,Ai-ONE 7S,2023,top20,2025,PGA Tour Stats / Eden Steak analysis (edenstea...
1,2,Taylor Montgomery,0.917,mallet,TaylorMade,Ghost Spider S,2012,top20,2025,PGA Tour Stats / Eden Steak analysis (edenstea...
2,3,Harry Hall,0.881,blade,Odyssey,O-Works Black #1 Wide S,2018,top20,2025,PGA Tour Stats / Eden Steak analysis (edenstea...
3,4,Denny McCarthy,0.679,mallet,Scotty Cameron,TOUR-only GoLo N7,2013,top20,2025,PGA Tour Stats / Eden Steak analysis (edenstea...
4,5,Nico Echavarria,0.666,mallet,Odyssey,Tri-Hot 5K Seven DB,2023,top20,2025,PGA Tour Stats / Eden Steak analysis (edenstea...


In [8]:
# Clean up player names
pga["player_name"] = pga["player_name"].str.strip().str.title()

# Standardize putter_type values
pga["putter_type"] = pga["putter_type"].str.strip().str.lower()

valid_types = {"blade", "mallet", "unknown"}
invalid_pga = pga[~pga["putter_type"].isin(valid_types)]

if not invalid_pga.empty:
    log(f"  WARNING: {len(invalid_pga)} rows had invalid putter_type values; setting them to 'unknown'")
    pga.loc[~pga["putter_type"].isin(valid_types), "putter_type"] = "unknown"

In [9]:
# Missing values
missing_pga = pga.isnull().sum()
log("  Missing values per column:")
if (missing_pga > 0).any():
    log(missing_pga[missing_pga > 0].to_string())
else:
    log("  No missing values found.")

# Deduplicate just in case
before_rows = len(pga)
pga = pga.drop_duplicates(subset=["player_name", "season"])
after_rows = len(pga)

log(f"  Deduplicated: {before_rows} → {after_rows} rows")

  Missing values per column:
  No missing values found.
  Deduplicated: 40 → 40 rows


In [10]:
# SG:Putting seasonal averages should be in a reasonable range
out_of_range_pga = pga[(pga["sg_putting_avg"] < -2.5) | (pga["sg_putting_avg"] > 2.5)]

if not out_of_range_pga.empty:
    bad_names = out_of_range_pga["player_name"].tolist()
    log(f"  WARNING: {len(out_of_range_pga)} rows have SG:Putting values outside [-2.5, 2.5]: {bad_names}")

# Add putter era classification
pga["putter_era"] = pga["putter_release_year"].apply(classify_era)

pga.head()

,rank,player_name,sg_putting_avg,putter_type,putter_brand,putter_model,putter_release_year,group,season,source,putter_era
0,1,Sam Burns,0.983,mallet,Odyssey,Ai-ONE 7S,2023,top20,2025,PGA Tour Stats / Eden Steak analysis (edenstea...,modern (2021+)
1,2,Taylor Montgomery,0.917,mallet,TaylorMade,Ghost Spider S,2012,top20,2025,PGA Tour Stats / Eden Steak analysis (edenstea...,classic (pre-2016)
2,3,Harry Hall,0.881,blade,Odyssey,O-Works Black #1 Wide S,2018,top20,2025,PGA Tour Stats / Eden Steak analysis (edenstea...,mid-era (2016-2020)
3,4,Denny Mccarthy,0.679,mallet,Scotty Cameron,TOUR-only GoLo N7,2013,top20,2025,PGA Tour Stats / Eden Steak analysis (edenstea...,classic (pre-2016)
4,5,Nico Echavarria,0.666,mallet,Odyssey,Tri-Hot 5K Seven DB,2023,top20,2025,PGA Tour Stats / Eden Steak analysis (edenstea...,modern (2021+)


In [11]:
clean_pga_path = os.path.join(CLEAN_DIR, "pga_sgputt_2025_clean.csv")
pga.to_csv(clean_pga_path, index=False)

log(f"  Saved: {clean_pga_path}  SHA-256={sha256(clean_pga_path)}")

  Saved: .\data\cleaned\pga_sgputt_2025_clean.csv  SHA-256=88403a4fe515cbcd7658ed1ebc5d422104f9421697c2b7ce0f2a2e16298910b0


## Dataset 2: Clean 2026 Masters leaderboard data

In [12]:
log("\n── Dataset 2: 2026 Masters Leaderboard ──")

masters_path = os.path.join(RAW_DIR, "masters_2026_leaderboard_raw.csv")
masters = pd.read_csv(masters_path)

log(f"Loaded {len(masters)} rows, {len(masters.columns)} columns")
masters.head()


── Dataset 2: 2026 Masters Leaderboard ──
Loaded 44 rows, 12 columns


,finish,player_name,score_to_par,r1,r2,r3,r4,putter_type,putter_brand,putter_model,putter_notes,source
0,1,Rory McIlroy,-13,67,65,73.0,71.0,mallet,TaylorMade,Spider Tour X,Switched at 2024 Tour Championship; used to wi...,Golf Monthly WITB 2026 / MyGolfSpy
1,2,Scottie Scheffler,-12,70,67,65.0,70.0,mallet,TaylorMade,Spider L-Neck,Switched from blade in 2024; credited with maj...,PGA Tour equipment reports / Golf Monthly
2,T3,Tyrrell Hatton,-11,69,66,77.0,66.0,mallet,Odyssey,Ai-One #7,Long-time Odyssey user; confirmed at 2026 Masters,Golf Monthly equipment coverage
3,T3,Justin Rose,-11,70,69,69.0,69.0,mallet,Scotty Cameron,Phantom X prototype,Switched from earlier models; confirmed via Go...,Golf Monthly WITB 2026
4,T3,Cameron Young,-11,72,68,65.0,72.0,mallet,Scotty Cameron,Phantom 9.5R Tour Prototype,Used to win 2025 Wyndham Championship; confirm...,Eden Steak / Golf Monthly


In [13]:
# Clean up player names
masters["player_name"] = masters["player_name"].str.strip().str.title()

# Standardize putter_type values
masters["putter_type"] = masters["putter_type"].str.strip().str.lower()

invalid_masters = masters[~masters["putter_type"].isin(valid_types)]

if not invalid_masters.empty:
    log(f"  WARNING: {len(invalid_masters)} rows had invalid putter_type values; setting them to 'unknown'")
    masters.loc[~masters["putter_type"].isin(valid_types), "putter_type"] = "unknown"

In [14]:
made_cut = masters[masters["finish"] != "MC"].copy()
missed_cut = masters[masters["finish"] == "MC"].copy()

log(f"  Made cut: {len(made_cut)} players | Missed cut: {len(missed_cut)} players")

  Made cut: 40 players | Missed cut: 4 players


In [15]:
def parse_finish(value):
    if value == "MC":
        return 999
    return int(str(value).replace("T", ""))


masters["finish_num"] = masters["finish"].apply(parse_finish)
masters["score_to_par"] = pd.to_numeric(masters["score_to_par"], errors="coerce")

In [16]:
missing_masters = masters.isnull().sum()
log("  Missing values per column:")
if (missing_masters > 0).any():
    log(missing_masters[missing_masters > 0].to_string())
else:
    log("  No missing values found.")

clean_masters_path = os.path.join(CLEAN_DIR, "masters_2026_clean.csv")
masters.to_csv(clean_masters_path, index=False)

log(f"  Saved: {clean_masters_path}  SHA-256={sha256(clean_masters_path)}")

  Missing values per column:
r3    4
r4    4
  Saved: .\data\cleaned\masters_2026_clean.csv  SHA-256=8b4fd7ea65ae01ca7c4bded360eb4098be034c62444c558868311cd50c7db7c7


## Integrate the two datasets

Here I merge the cleaned PGA season data with the Masters tournament data using player name as the shared field.

In [17]:
pga_merge = pga[
    [
        "player_name",
        "rank",
        "sg_putting_avg",
        "putter_type",
        "putter_brand",
        "putter_model",
        "group",
        "season",
    ]
].copy()

pga_merge = pga_merge.rename(
    columns={
        "rank": "pga_2025_rank",
        "sg_putting_avg": "pga_2025_sg_putt",
        "putter_type": "putter_type_2025",
        "putter_brand": "putter_brand_2025",
        "putter_model": "putter_model_2025",
        "group": "pga_2025_group",
    }
)

pga_merge.head()

,player_name,pga_2025_rank,pga_2025_sg_putt,putter_type_2025,putter_brand_2025,putter_model_2025,pga_2025_group,season
0,Sam Burns,1,0.983,mallet,Odyssey,Ai-ONE 7S,top20,2025
1,Taylor Montgomery,2,0.917,mallet,TaylorMade,Ghost Spider S,top20,2025
2,Harry Hall,3,0.881,blade,Odyssey,O-Works Black #1 Wide S,top20,2025
3,Denny Mccarthy,4,0.679,mallet,Scotty Cameron,TOUR-only GoLo N7,top20,2025
4,Nico Echavarria,5,0.666,mallet,Odyssey,Tri-Hot 5K Seven DB,top20,2025


In [18]:
masters_merge = masters[masters["finish"] != "MC"][
    [
        "player_name",
        "finish",
        "finish_num",
        "score_to_par",
        "putter_type",
        "putter_brand",
        "putter_model",
    ]
].copy()

masters_merge = masters_merge.rename(
    columns={
        "finish": "masters_2026_finish",
        "finish_num": "masters_2026_finish_num",
        "score_to_par": "masters_2026_score",
        "putter_type": "putter_type_masters",
        "putter_brand": "putter_brand_masters",
        "putter_model": "putter_model_masters",
    }
)

masters_merge.head()

,player_name,masters_2026_finish,masters_2026_finish_num,masters_2026_score,putter_type_masters,putter_brand_masters,putter_model_masters
0,Rory Mcilroy,1,1,-13,mallet,TaylorMade,Spider Tour X
1,Scottie Scheffler,2,2,-12,mallet,TaylorMade,Spider L-Neck
2,Tyrrell Hatton,T3,3,-11,mallet,Odyssey,Ai-One #7
3,Justin Rose,T3,3,-11,mallet,Scotty Cameron,Phantom X prototype
4,Cameron Young,T3,3,-11,mallet,Scotty Cameron,Phantom 9.5R Tour Prototype


In [19]:
# Full outer join keeps everyone from both datasets
integrated = pd.merge(pga_merge, masters_merge, on="player_name", how="outer")

log(f"  Integrated dataset: {len(integrated)} player rows")
integrated.head()

  Integrated dataset: 75 player rows


,player_name,pga_2025_rank,pga_2025_sg_putt,putter_type_2025,putter_brand_2025,putter_model_2025,pga_2025_group,season,masters_2026_finish,masters_2026_finish_num,masters_2026_score,putter_type_masters,putter_brand_masters,putter_model_masters
0,Adam Scott,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T18,18.0,-5.0,mallet,L.A.B.,MEZZ.1 Max
1,Adam Svensson,173.0,-0.579,blade,Odyssey,Toulon Design 904L,bottom20,2025.0,NaN,NaN,NaN,NaN,NaN,NaN
2,Akshay Bhatia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,T18,18.0,-5.0,mallet,Ping,PLD Tyne 4
3,Alejandro Tosti,174.0,-0.580,blade,Scotty Cameron,Squareback,bottom20,2025.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Andrew Putnam,16.0,0.439,mallet,Odyssey,White Hot RX Rossie,top20,2025.0,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
def putter_consistency(row):
    putter_2025 = row.get("putter_type_2025")
    putter_masters = row.get("putter_type_masters")
    
    if pd.isna(putter_2025) or pd.isna(putter_masters):
        return "only_in_one_dataset"
    if putter_2025 == putter_masters:
        return "consistent"
    return f"changed: {putter_2025}→{putter_masters}"


integrated["putter_consistency"] = integrated.apply(putter_consistency, axis=1)

# Use Masters putter type when available because it is more recent
integrated["putter_type_unified"] = integrated["putter_type_masters"].fillna(
    integrated["putter_type_2025"]
)

In [21]:
players_in_both = (
    integrated["putter_type_2025"].notna() & integrated["putter_type_masters"].notna()
).sum()

only_in_pga = (
    integrated["putter_type_2025"].notna() & integrated["putter_type_masters"].isna()
).sum()

only_in_masters = (
    integrated["putter_type_2025"].isna() & integrated["putter_type_masters"].notna()
).sum()

log(f"  Players in both datasets: {players_in_both}")
log(f"  Only in PGA 2025 data: {only_in_pga}")
log(f"  Only in Masters data:  {only_in_masters}")

log("\n  Putter consistency breakdown:")
log(integrated["putter_consistency"].value_counts().to_string())

  Players in both datasets: 5
  Only in PGA 2025 data: 35
  Only in Masters data:  35

  Putter consistency breakdown:
putter_consistency
only_in_one_dataset    70
consistent              5


In [22]:
integrated_path = os.path.join(INT_DIR, "integrated_putter_analysis.csv")
integrated.to_csv(integrated_path, index=False)

log(f"\n  Saved: {integrated_path}  SHA-256={sha256(integrated_path)}")


  Saved: .\data\integrated\integrated_putter_analysis.csv  SHA-256=b76531990cbe4bb93b0d505c912e8b37659d5dd4b9531f0e42ad1b86a2293db0


## Summary statistics

These checks give a quick sense of how putter types are distributed across the cleaned and integrated data.

In [23]:
log("\n── Summary Statistics ──")

log("\n[PGA 2025 – Top 20 putter type breakdown]")
top20 = pga[pga["group"] == "top20"]

log(top20["putter_type"].value_counts().to_string())
log("  Top 20 mean SG:Putt by type:")
log(top20.groupby("putter_type")["sg_putting_avg"].mean().round(3).to_string())


── Summary Statistics ──

[PGA 2025 – Top 20 putter type breakdown]
putter_type
mallet    17
blade      3
  Top 20 mean SG:Putt by type:
putter_type
blade     0.578
mallet    0.585


In [24]:
log("\n[PGA 2025 – Bottom 20 putter type breakdown]")
bottom20 = pga[pga["group"] == "bottom20"]

log(bottom20["putter_type"].value_counts().to_string())
log("  Bottom 20 mean SG:Putt by type:")
log(bottom20.groupby("putter_type")["sg_putting_avg"].mean().round(3).to_string())


[PGA 2025 – Bottom 20 putter type breakdown]
putter_type
mallet    11
blade      9
  Bottom 20 mean SG:Putt by type:
putter_type
blade    -0.577
mallet   -0.553


In [25]:
log("\n[Masters 2026 – made cut, putter type breakdown]")
masters_made_cut = masters[masters["finish"] != "MC"]
log(masters_made_cut["putter_type"].value_counts().to_string())

log("\n[Masters 2026 – top 10 finishers putter type]")
top10_masters = masters[masters["finish_num"] <= 10]
log(top10_masters[["player_name", "finish", "putter_type", "putter_model"]].to_string(index=False))


[Masters 2026 – made cut, putter type breakdown]
putter_type
mallet    30
blade     10

[Masters 2026 – top 10 finishers putter type]
      player_name finish putter_type                     putter_model
     Rory Mcilroy      1      mallet                    Spider Tour X
Scottie Scheffler      2      mallet                    Spider L-Neck
   Tyrrell Hatton     T3      mallet                        Ai-One #7
      Justin Rose     T3      mallet              Phantom X prototype
    Cameron Young     T3      mallet      Phantom 9.5R Tour Prototype
   Russell Henley     T3      mallet         Phantom 5 Tour Prototype
  Collin Morikawa     T7       blade Scotty Cameron Newport prototype
        Sam Burns     T7      mallet                       Ai-One #7S
         Max Homa     T9      mallet                 Ai-One Milled #7
Xander Schauffele     T9      mallet                     Phantom X 12


In [26]:
log_path = os.path.join(LOG_DIR, "clean_integrate_log.txt")

with open(log_path, "w", encoding="utf-8") as f:
    f.write("\n".join(log_lines))

print(f"\nLog written → {log_path}")


Log written → .\logs\clean_integrate_log.txt
